## ایمپورت کتابخانه های پایه

In [1]:
import os
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.utils import image_dataset_from_directory

warnings.filterwarnings("ignore")

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


## بررسی وجود GPU

## بررسی اطلاعات بیلد تنسورفلو

In [2]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU Found")
    print(gpus)
else:
    print("Running on CPU")

print(tf.sysconfig.get_build_info())

Running on CPU
OrderedDict({'is_cuda_build': False, 'is_rocm_build': False, 'is_tensorrt_build': False, 'msvcp_dll_names': 'msvcp140.dll,msvcp140_1.dll'})


## تعریف ابرپارامترها

In [1]:
IMG_SIZE = (224,224)

BATCH_SIZE = 32

EPOCHS_TRANSFER = 5

SEED = 42

## تعریف مسیرهای دیتاست

In [4]:
DATASET_PATH = "Dataset"

TRAIN_PATH = os.path.join(DATASET_PATH,"train")

VALID_PATH = os.path.join(DATASET_PATH,"valid")

print(TRAIN_PATH)
print(VALID_PATH)

Dataset\train
Dataset\valid


## بازرسی ساختار پوشه های دیتاست

## شمارش تصاویر هر کلاس

In [5]:
for root, dirs, files in os.walk(DATASET_PATH):

    print(root)

    print("Folders:",dirs)

    print("Images:",len(files))

    print("-"*50)

    print()

mel_train = len(os.listdir(os.path.join(TRAIN_PATH,"Melanoma")))
not_train = len(os.listdir(os.path.join(TRAIN_PATH,"NotMelanoma")))

mel_valid = len(os.listdir(os.path.join(VALID_PATH,"Melanoma")))
not_valid = len(os.listdir(os.path.join(VALID_PATH,"NotMelanoma")))

print("Train Melanoma :",mel_train)
print("Train NotMelanoma :",not_train)

print()

print("Validation Melanoma :",mel_valid)
print("Validation NotMelanoma :",not_valid)

Dataset
Folders: ['train', 'valid']
Images: 0
--------------------------------------------------

Dataset\train
Folders: ['Melanoma', 'NotMelanoma']
Images: 0
--------------------------------------------------

Dataset\train\Melanoma
Folders: []
Images: 5341
--------------------------------------------------

Dataset\train\NotMelanoma
Folders: []
Images: 5341
--------------------------------------------------

Dataset\valid
Folders: ['Melanoma', 'NotMelanoma']
Images: 0
--------------------------------------------------

Dataset\valid\Melanoma
Folders: []
Images: 1781
--------------------------------------------------

Dataset\valid\NotMelanoma
Folders: []
Images: 1781
--------------------------------------------------

Train Melanoma : 5341
Train NotMelanoma : 5341

Validation Melanoma : 1781
Validation NotMelanoma : 1781


## ساخت دیتاست آموزش

In [6]:
train_dataset = image_dataset_from_directory(
    TRAIN_PATH,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "binary",
    shuffle = True,
    seed = SEED
)


print("Train dataset created")

Found 10682 files belonging to 2 classes.
Train dataset created


## ساخت دیتاست اعتبارسنجی

In [7]:
valid_dataset = image_dataset_from_directory(
    VALID_PATH,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "binary",
    shuffle = False
)


print("Validation dataset created")

Found 3562 files belonging to 2 classes.
Validation dataset created


## نام کلاس ها و قرارداد برچسب ها

In [8]:
class_names = train_dataset.class_names

print(class_names)

['Melanoma', 'NotMelanoma']


## تعریف لایه های افزایش داده

## اعمال Augmentation روی دیتاست آموزش

In [9]:
data_augmentation = models.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1),

    layers.RandomContrast(0.1)

])


print("Augmentation ready")

print()

train_dataset = train_dataset.map(
    lambda x,y: (data_augmentation(x,training=True),y)
)


print("Augmentation applied")

Augmentation ready

Augmentation applied


## بهینه سازی خط لوله داده

In [10]:
AUTOTUNE = tf.data.AUTOTUNE


train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)


valid_dataset = valid_dataset.prefetch(
    buffer_size=AUTOTUNE
)


print("Pipeline optimized")

Pipeline optimized


## ساخت مدل MobileNetV2 ( مدل سبک پروژه)

In [12]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam


def build_mobilenet_model():

    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )

    base_model.trainable = False


    model = models.Sequential([
        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.3),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.Dropout(0.2),

        layers.Dense(
            1,
            activation="sigmoid"
        )
    ])


    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )


    return model



mobile_model = build_mobilenet_model()


mobile_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## آموزش MobileNetV2__مرحله یادگیری انتقالی

In [13]:
history_mobile = mobile_model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=5
)

Epoch 1/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 335s 986ms/step - accuracy: 0.5967 - loss: 0.6741 - val_accuracy: 0.7892 - val_loss: 0.4753
Epoch 2/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 284s 850ms/step - accuracy: 0.6843 - loss: 0.5872 - val_accuracy: 0.7131 - val_loss: 0.5223
Epoch 3/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 281s 839ms/step - accuracy: 0.7230 - loss: 0.5433 - val_accuracy: 0.7181 - val_loss: 0.5419
Epoch 4/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 275s 822ms/step - accuracy: 0.7439 - loss: 0.5213 - val_accuracy: 0.8130 - val_loss: 0.4351
Epoch 5/5
334/334 ━━━━━━━━━━━━━━━━━━━━ 276s 825ms/step - accuracy: 0.7531 - loss: 0.5057 - val_accuracy: 0.7305 - val_loss: 0.5522


## باز کردن بدنه MobileNetV2 برای تنظیم دقیق

## فریز کردن همه لایه ها به جز 30 لایه آخر

In [14]:
mobile_model.layers[0].trainable = True

for layer in mobile_model.layers[0].layers[:-30]:
    layer.trainable=False

## کامپایل مجدد با نرخ یادگیری کوچک

In [15]:
mobile_model.compile(
optimizer=tf.keras.optimizers.Adam(1e-5),
loss="binary_crossentropy",
metrics=["accuracy"]
)

## تنظیم دقیق MobileNetV2 (10 اپوک)

In [16]:
history_mobile_ft = mobile_model.fit(
train_dataset,
validation_data=valid_dataset,
epochs=10
)

Epoch 1/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 329s 946ms/step - accuracy: 0.7492 - loss: 0.5136 - val_accuracy: 0.7075 - val_loss: 0.6417
Epoch 2/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 317s 949ms/step - accuracy: 0.8184 - loss: 0.4063 - val_accuracy: 0.6802 - val_loss: 0.6948
Epoch 3/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 322s 964ms/step - accuracy: 0.8440 - loss: 0.3550 - val_accuracy: 0.6707 - val_loss: 0.7144
Epoch 4/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 313s 937ms/step - accuracy: 0.8520 - loss: 0.3415 - val_accuracy: 0.7277 - val_loss: 0.6123
Epoch 5/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 338s 1s/step - accuracy: 0.8631 - loss: 0.3176 - val_accuracy: 0.7033 - val_loss: 0.6874
Epoch 6/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 309s 924ms/step - accuracy: 0.8673 - loss: 0.3076 - val_accuracy: 0.7844 - val_loss: 0.5224
Epoch 7/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 306s 916ms/step - accuracy: 0.8725 - loss: 0.2968 - val_accuracy: 0.8007 - val_loss: 0.4817
Epoch 8/10
334/334 ━━━━━━━━━━━━━━━━━━━━ 311s 930ms/step - accuracy: 0.8776 - lo

## ذخیره مدل MobileNetV2

In [17]:
mobile_model.save("MobileNetV2.keras")